# 🟠 Notebook 2 — Pipeline de Extracción Estructurada
> **Del texto caótico a datos limpios**

En este notebook vamos a:
1. Usar Gemini via LangChain para extraer datos estructurados (JSON) desde texto libre
2. Alimentar esos datos a un modelo ML downstream
3. Ver cómo el LLM actúa como **paso de preprocesamiento** en el pipeline

**Módulo:** IA Generativa — Clase 4: Pipelines ML + GenAI

## 1. Instalación de dependencias

In [1]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 20.6 MB/s eta 0:00:00


In [3]:
!pip install -q transformers accelerate torch

## 2. Configuración

In [13]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
# from langchain.schema.output_parser import StrOutputParser
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

load_dotenv()
# GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

print("✅ LLM configurado")

✅ LLM configurado


## 3. Dataset de emails / tickets de soporte (texto libre)

In [14]:
tickets_texto = [
    """Hola, soy María González. Llevo 3 días sin poder acceder a mi cuenta premium.
    Intenté restablecer la contraseña pero el email no llega. Estoy muy frustrada,
    ya pagué este mes y no puedo usar el servicio. Necesito solución urgente.""",

    """Buenos días. Mi nombre es Carlos López. Quería consultar si es posible
    hacer upgrade a plan enterprise para mi empresa. Somos 15 personas y el plan
    actual nos queda chico. ¿Tienen descuentos por volumen?""",

    """Hola! Soy Ana Martínez. Encontré un bug en la aplicación móvil:
    cuando intento exportar a PDF en iOS se cierra la app. Pasa siempre
    en iPhone 14 con iOS 17. Adjunto el log de error.""",

    """Buenas tardes. Soy Roberto Silva. Estoy pensando en cancelar mi suscripción
    porque el precio subió mucho y no uso todas las features. ¿Tienen algún plan
    más económico o descuento por lealtad? Llevo 2 años con ustedes.""",

    """Hola, Pedro Ramírez aquí. El cobro de este mes fue incorrecto,
    me cobraron $150 pero debería ser $80 según mi plan básico.
    Necesito que revisen y hagan el reintegro lo antes posible.""",

    """Buenas! Laura Fernández. Me encanta el producto, funciona genial.
    Solo quería sugerir que agreguen integración con Notion porque lo uso
    mucho para trabajar. Sería increíble tenerlo todo conectado.""",
]

print(f"Tickets a procesar: {len(tickets_texto)}")
print("\nEjemplo de ticket:")
print(tickets_texto[0])

Tickets a procesar: 6

Ejemplo de ticket:
Hola, soy María González. Llevo 3 días sin poder acceder a mi cuenta premium.
    Intenté restablecer la contraseña pero el email no llega. Estoy muy frustrada,
    ya pagué este mes y no puedo usar el servicio. Necesito solución urgente.


## 4. Extracción estructurada con LangChain + Gemini

In [15]:
prompt_extraccion = ChatPromptTemplate.from_template("""
Analiza el siguiente ticket de soporte y extrae la información en formato JSON.
Responde ÚNICAMENTE con el JSON, sin explicaciones ni markdown.

Ticket:
{ticket}

Extrae estos campos:
{{
  "nombre_cliente": "nombre completo o null",
  "categoria": "uno de: bug_tecnico | consulta_comercial | facturacion | cancelacion | sugerencia | acceso_cuenta",
  "urgencia": "uno de: alta | media | baja",
  "sentimiento": "uno de: muy_negativo | negativo | neutro | positivo | muy_positivo",
  "plan_actual": "básico | estándar | premium | enterprise | desconocido",
  "requiere_accion_inmediata": true o false,
  "resumen": "máximo 15 palabras"
}}
""")

chain_extraccion = prompt_extraccion | llm | StrOutputParser()

def extraer_datos(ticket_texto: str) -> dict:
    """Extrae datos estructurados de un ticket de texto libre."""
    raw = chain_extraccion.invoke({"ticket": ticket_texto})
    # Limpiar posibles backticks de markdown
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

# Procesar todos los tickets
tickets_estructurados = []
for i, ticket in enumerate(tickets_texto):
    print(f"Procesando ticket {i+1}/{len(tickets_texto)}...", end=" ")
    datos = extraer_datos(ticket)
    datos["texto_original"] = ticket
    tickets_estructurados.append(datos)
    print("✅")

df_tickets = pd.DataFrame(tickets_estructurados)
print("\nDatos extraídos:")
df_tickets[["nombre_cliente", "categoria", "urgencia", "sentimiento", "requiere_accion_inmediata", "resumen"]]

Procesando ticket 1/6... ✅
Procesando ticket 2/6... ✅
Procesando ticket 3/6... ✅
Procesando ticket 4/6... ✅
Procesando ticket 5/6... ✅
Procesando ticket 6/6... ✅

Datos extraídos:


,nombre_cliente,categoria,urgencia,sentimiento,requiere_accion_inmediata,resumen
0,María González,acceso_cuenta,alta,muy_negativo,True,"3 días sin acceso a cuenta premium, email de r..."
1,Carlos López,consulta_comercial,media,positivo,False,"Consulta upgrade plan enterprise, descuentos p..."
2,Ana Martínez,bug_tecnico,alta,negativo,True,App móvil se cierra al exportar a PDF en iOS 17.
3,Roberto Silva,cancelacion,media,negativo,True,Cliente considera cancelar suscripción por pre...
4,Pedro Ramírez,facturacion,alta,negativo,True,Cobro incorrecto de $150 en lugar de $80. Soli...
5,Laura Fernández,sugerencia,baja,muy_positivo,False,Sugerencia de integración con Notion para cone...


## 5. Análisis exploratorio de los datos extraídos

In [16]:
print("=" * 50)
print("📊 DISTRIBUCIÓN DE CATEGORÍAS")
print("=" * 50)
print(df_tickets["categoria"].value_counts().to_string())

print("\n" + "=" * 50)
print("😊 DISTRIBUCIÓN DE SENTIMIENTO")
print("=" * 50)
print(df_tickets["sentimiento"].value_counts().to_string())

print("\n" + "=" * 50)
print("🚨 TICKETS QUE REQUIEREN ACCIÓN INMEDIATA")
print("=" * 50)
urgentes = df_tickets[df_tickets["requiere_accion_inmediata"] == True]
for _, row in urgentes.iterrows():
    print(f"  • {row['nombre_cliente']}: {row['resumen']} ({row['urgencia']})")

📊 DISTRIBUCIÓN DE CATEGORÍAS
categoria
acceso_cuenta         1
consulta_comercial    1
bug_tecnico           1
cancelacion           1
facturacion           1
sugerencia            1

😊 DISTRIBUCIÓN DE SENTIMIENTO
sentimiento
negativo        3
muy_negativo    1
positivo        1
muy_positivo    1

🚨 TICKETS QUE REQUIEREN ACCIÓN INMEDIATA
  • María González: 3 días sin acceso a cuenta premium, email de restablecimiento no llega. (alta)
  • Ana Martínez: App móvil se cierra al exportar a PDF en iOS 17. (alta)
  • Roberto Silva: Cliente considera cancelar suscripción por precio y falta de uso, busca plan económico o descuento. (media)
  • Pedro Ramírez: Cobro incorrecto de $150 en lugar de $80. Solicita revisión y reintegro. (alta)


## 6. Pipeline downstream: Priorización automática con ML

In [17]:
from sklearn.preprocessing import LabelEncoder

# Encodear variables categóricas para el modelo
df_ml = df_tickets.copy()

encoders = {}
for col in ["categoria", "sentimiento", "urgencia", "plan_actual"]:
    le = LabelEncoder()
    df_ml[f"{col}_enc"] = le.fit_transform(df_ml[col].fillna("desconocido"))
    encoders[col] = le

df_ml["requiere_accion_num"] = df_ml["requiere_accion_inmediata"].astype(int)

# Score de prioridad = combinación ponderada de factores
df_ml["score_prioridad"] = (
    df_ml["requiere_accion_num"] * 40 +
    (3 - df_ml["urgencia_enc"]) * 20 +   # alta urgencia = menor enc value
    (2 - df_ml["sentimiento_enc"].clip(0,2)) * 15 +
    df_ml["categoria_enc"] * 5
).astype(int)

# Nivel de prioridad
df_ml["nivel_prioridad"] = pd.cut(
    df_ml["score_prioridad"],
    bins=[-1, 20, 45, 100],
    labels=["🟢 Normal", "🟡 Prioritario", "🔴 Urgente"]
)

resultado = df_ml[["nombre_cliente", "categoria", "sentimiento",
                    "urgencia", "score_prioridad", "nivel_prioridad"]]\
              .sort_values("score_prioridad", ascending=False)

print("\n📋 QUEUE DE TICKETS PRIORIZADO:")
print(resultado.to_string(index=False))


📋 QUEUE DE TICKETS PRIORIZADO:
 nombre_cliente          categoria  sentimiento urgencia  score_prioridad nivel_prioridad
 María González      acceso_cuenta muy_negativo     alta              130             NaN
  Pedro Ramírez        facturacion     negativo     alta              120             NaN
   Ana Martínez        bug_tecnico     negativo     alta              105             NaN
Laura Fernández         sugerencia muy_positivo     baja               80       🔴 Urgente
  Roberto Silva        cancelacion     negativo    media               70       🔴 Urgente
   Carlos López consulta_comercial     positivo    media               35   🟡 Prioritario


## 7. Generación automática de respuesta inicial

In [18]:
prompt_respuesta = ChatPromptTemplate.from_template("""
Eres un agente de soporte empático y profesional.
Genera un primer mensaje de respuesta para este ticket.

Cliente: {nombre_cliente}
Categoría: {categoria}
Sentimiento del cliente: {sentimiento}
Urgencia: {urgencia}
Resumen del problema: {resumen}
Prioridad asignada: {nivel_prioridad}

Genera una respuesta inicial de 3-4 oraciones que:
- Reconozca el problema específico del cliente
- Sea empática según el sentimiento detectado
- Indique el próximo paso concreto
- No prometa tiempos que no puedas cumplir
""")

chain_respuesta = prompt_respuesta | llm | StrOutputParser()

# Responder al ticket más urgente
ticket_urgente = df_ml.loc[df_ml["score_prioridad"].idxmax()]

respuesta = chain_respuesta.invoke({
    "nombre_cliente": ticket_urgente["nombre_cliente"],
    "categoria": ticket_urgente["categoria"],
    "sentimiento": ticket_urgente["sentimiento"],
    "urgencia": ticket_urgente["urgencia"],
    "resumen": ticket_urgente["resumen"],
    "nivel_prioridad": str(ticket_urgente["nivel_prioridad"])
})

print(f"\n📧 RESPUESTA AUTOMÁTICA PARA: {ticket_urgente['nombre_cliente']}")
print("-" * 50)
print(respuesta)


📧 RESPUESTA AUTOMÁTICA PARA: María González
--------------------------------------------------
Hola María,

Lamento muchísimo la situación tan frustrante que estás viviendo al llevar 3 días sin acceso a tu cuenta premium y al no recibir el email de restablecimiento. Comprendo perfectamente lo importante que es para ti tener acceso a tu cuenta y la molestia que esto te debe estar causando. Para poder resolver esto cuanto antes, voy a investigar de inmediato qué está ocurriendo con tu cuenta y el envío de correos. Te contactaré con una actualización tan pronto como tenga más información sobre el avance.


## 🎯 Conclusiones

**Patrón clave:** `Texto libre → LLM (extracción) → JSON estructurado → ML (priorización) → LLM (respuesta)`

- El LLM resuelve el problema de datos **no estructurados** que los modelos ML no pueden procesar
- Una vez estructurados, los datos fluyen naturalmente hacia modelos, dashboards o bases de datos
- Este pipeline se puede escalar a miles de tickets con mínimo esfuerzo